# 14. Language Models in NLP

## Overview

A **language model (LM)** assigns probabilities to sequences of tokens. It estimates how likely a sequence is in a language and can be used to predict a likely next token.

This notebook introduces statistical language modeling through n-gram models, probability estimation, and smoothing. It uses small, transparent examples to make the calculations easy to inspect.

## Learning Objectives

By the end of this notebook, you will understand:
- What a language model estimates.
- The difference between unigram and bigram models.
- How maximum-likelihood probabilities are calculated from counts.
- Why unseen n-grams create a problem.
- How Laplace (add-one) smoothing addresses zero probabilities.
- How to calculate sentence probability with a simple bigram model.

## 1. What Does a Language Model Do?

A language model estimates the probability of a token sequence. For a sequence of tokens, the chain rule gives:

$$P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \ldots, w_{i-1})$$

In practice, estimating every possible history is difficult. An n-gram model simplifies the context by using only the previous `n - 1` tokens.

- **Unigram model:** treats each token independently.
- **Bigram model:** estimates a token from the immediately preceding token.
- **Trigram model:** estimates a token from the two preceding tokens.

In [1]:
sentence = "language models learn patterns from text"
tokens = sentence.lower().split()

print("Tokens:", tokens)
print("Number of tokens:", len(tokens))

Tokens: ['language', 'models', 'learn', 'patterns', 'from', 'text']
Number of tokens: 6


## 2. Build a Small Training Corpus

A statistical language model learns probabilities from a corpus. The following toy corpus contains repeated word sequences so that the estimated probabilities are easy to interpret.

Each sentence is treated as a separate sequence. We add `<s>` and `</s>` markers to represent sentence beginnings and endings.

In [2]:
corpus = [
    "the cat sat",
    "the cat ate fish",
    "the dog sat",
    "the dog ate fish",
]

def tokenize_with_boundaries(sentence):
    """Tokenize a sentence and add start/end markers."""
    return ["<s>"] + sentence.lower().split() + ["</s>"]

tokenized_corpus = [tokenize_with_boundaries(sentence) for sentence in corpus]

for tokens in tokenized_corpus:
    print(tokens)

['<s>', 'the', 'cat', 'sat', '</s>']
['<s>', 'the', 'cat', 'ate', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', '</s>']
['<s>', 'the', 'dog', 'ate', 'fish', '</s>']


## 3. Unigram Language Model

A unigram model estimates token probability using its relative frequency:

$$P(w) = \frac{C(w)}{N}$$

Here, `C(w)` is the number of occurrences of token `w`, and `N` is the total number of tokens in the training corpus.

In [3]:
from collections import Counter

unigram_counts = Counter(
    token
    for sentence_tokens in tokenized_corpus
    for token in sentence_tokens
)

total_tokens = sum(unigram_counts.values())
unigram_probabilities = {
    token: count / total_tokens
    for token, count in unigram_counts.items()
}

print("Total token count:", total_tokens)
print("P('the'):", round(unigram_probabilities.get("the", 0), 4))
print("P('cat'):", round(unigram_probabilities.get("cat", 0), 4))

Total token count: 22
P('the'): 0.1818
P('cat'): 0.0909


## 4. Bigram Language Model

A bigram model estimates the probability of the current token given the previous token:

$$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$

The numerator counts the bigram, while the denominator counts occurrences of the preceding token as a context. Sentence boundaries are included in the counts.

In [4]:
from nltk.util import bigrams

bigram_counts = Counter(
    bigram
    for sentence_tokens in tokenized_corpus
    for bigram in bigrams(sentence_tokens)
)

context_counts = Counter(
    token
    for sentence_tokens in tokenized_corpus
    for token in sentence_tokens[:-1]
)

def bigram_probability(previous_token, current_token):
    """Return the maximum-likelihood probability for a bigram."""
    denominator = context_counts[previous_token]
    if denominator == 0:
        return 0.0
    return bigram_counts[(previous_token, current_token)] / denominator

print("P(cat | the):", round(bigram_probability("the", "cat"), 4))
print("P(dog | the):", round(bigram_probability("the", "dog"), 4))
print("P(fish | cat):", round(bigram_probability("cat", "fish"), 4))

P(cat | the): 0.5
P(dog | the): 0.5
P(fish | cat): 0.0


For this corpus, `P(cat | the)` and `P(dog | the)` are both `0.5`, because `the` is followed by `cat` twice and by `dog` twice. `P(fish | cat)` is zero because that bigram does not occur in the corpus.

## 5. The Zero-Probability Problem

Maximum-likelihood estimation assigns probability zero to any unseen bigram. If a sentence contains even one zero-probability bigram, its entire product probability becomes zero.

This is a major issue because a valid phrase may be absent from a small training corpus.

In [5]:
observed_bigram = ("the", "cat")
unseen_bigram = ("cat", "fish")

print(observed_bigram, "->", bigram_probability(*observed_bigram))
print(unseen_bigram, "->", bigram_probability(*unseen_bigram))

('the', 'cat') -> 0.5
('cat', 'fish') -> 0.0


## 6. Laplace (Add-One) Smoothing

Laplace smoothing adds one to each possible continuation count:

$$P_{\text{Laplace}}(w_i \mid w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}$$

`V` is the vocabulary size. The method gives unseen events a non-zero probability, though it can overestimate rare events in realistic corpora.

In [6]:
vocabulary = set(
    token
    for sentence_tokens in tokenized_corpus
    for token in sentence_tokens
)
vocabulary_size = len(vocabulary)

def laplace_bigram_probability(previous_token, current_token):
    """Return an add-one-smoothed bigram probability."""
    numerator = bigram_counts[(previous_token, current_token)] + 1
    denominator = context_counts[previous_token] + vocabulary_size
    return numerator / denominator

print("Unsmoothed P(fish | cat):", bigram_probability("cat", "fish"))
print("Laplace-smoothed P(fish | cat):", round(laplace_bigram_probability("cat", "fish"), 4))

Unsmoothed P(fish | cat): 0.0
Laplace-smoothed P(fish | cat): 0.1


## 7. Estimate a Sentence Probability

A bigram model approximates a sentence probability by multiplying conditional probabilities for each token given its predecessor. We use log probabilities for numerical stability when sequences become longer.

The function below uses Laplace-smoothed probabilities so that unseen bigrams do not make the whole result zero.

In [7]:
import math
from nltk.util import bigrams

def sentence_log_probability(sentence):
    """Calculate a Laplace-smoothed bigram log probability."""
    sentence_tokens = tokenize_with_boundaries(sentence)
    log_probability = 0.0

    for previous_token, current_token in bigrams(sentence_tokens):
        probability = laplace_bigram_probability(previous_token, current_token)
        log_probability += math.log(probability)

    return log_probability

sentences_to_compare = [
    "the cat sat",
    "the dog ate fish",
    "the fish sat",
]

for sentence in sentences_to_compare:
    print(f"{sentence!r}: {sentence_log_probability(sentence):.4f}")

'the cat sat': -5.0752
'the dog ate fish': -6.2791
'the fish sat': -6.8669


A larger log probability (closer to zero) indicates a higher model-assigned probability for the sentence. These scores are meaningful only relative to the same model and preprocessing setup.

## 8. Inspect the Most Frequent Bigrams

Frequent bigrams summarize local patterns in the training corpus. They are descriptive statistics, not necessarily a complete language model.

In [8]:
for bigram, count in bigram_counts.most_common():
    print(f"{' '.join(bigram):<15} {count}")

<s> the         4
the cat         2
sat </s>        2
ate fish        2
fish </s>       2
the dog         2
cat sat         1
cat ate         1
dog sat         1
dog ate         1


## 9. Strengths and Limitations

### Strengths
- Simple to implement and interpret.
- Can be trained from token counts without a neural network.
- Useful as a baseline for text prediction and sequence scoring.

### Limitations
- **Data sparsity:** many possible sequences are unseen.
- **Limited context:** an n-gram model only uses a short fixed history.
- **Vocabulary sensitivity:** unseen words need explicit handling.
- **Smoothing trade-offs:** simple smoothing may distribute too much probability to rare events.
- **No semantic understanding:** word-count patterns alone do not represent meaning the way modern contextual models attempt to.

## 10. Summary

- A language model assigns probabilities to token sequences.
- Unigram models use token frequencies; bigram models condition on the previous token.
- Maximum-likelihood estimates can assign zero probability to unseen n-grams.
- Laplace smoothing provides non-zero estimates for unseen continuations.
- Sentence log probability allows sequence scores to be compared more safely than raw products.
- Statistical n-gram models are useful baselines, but they have limited context and can suffer from data sparsity.